# 🛰️🔆 TKG Solar — Colab Training (DKASC Alice Springs)

Tái lập **TKGSolarModel** + baselines trên dữ liệu co-located **DKASC Alice Springs (2020–2022) + Himawari-8**.

## 🗓️ KẾ HOẠCH 2 PHIÊN
> Kết quả mỗi phiên **tự lưu Drive** → bảng tổng (B4) ghép được dù train ở 2 phiên/2 GPU khác nhau. `AUTO_RESUME` + checkpoint Drive → đứt session chạy lại là tiếp tục.

**🟧 Proposed (nặng) — A100 40GB:** §1→§6 → **B1 → B2 → B3** (KHÔNG chạy Phase A để khỏi tốn CU).
**🟦 Baselines — T4 free:** §1→§6 → **A1 → A2 → A3**.
**Bảng tổng:** chạy **B4** (tự nạp cả 2 từ Drive).

Upload Drive: `dkasc/alice_2020_2022_clean.csv` (18MB, cả 2) + `himawari_alice/frames.h5` (1.2GB, chỉ Proposed).

<hr style="border:none;border-top:3px solid #d95f0e">
## 1 · GPU

In [1]:
!nvidia-smi

Sun Jun 28 08:09:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             49W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

<hr style="border:none;border-top:3px solid #d95f0e">
## 2 · Clone repo (nhánh `init-project`)

In [2]:
import os
if not os.path.isdir('tkg-solar-satellite-reasoning'):
    !git clone https://github.com/DucLong06/tkg-solar-satellite-reasoning.git
%cd tkg-solar-satellite-reasoning
!git checkout init-project && git pull

Cloning into 'tkg-solar-satellite-reasoning'...
remote: Enumerating objects: 255, done.
remote: Counting objects: 100% (255/255), done.
remote: Compressing objects: 100% (162/162), done.
Receiving objects: 100% (255/255), 289.31 KiB | 22.25 MiB/s, done.
remote: Total 255 (delta 139), reused 203 (delta 90), pack-reused 0 (from 0)
Resolving deltas: 100% (139/139), done.
/content/tkg-solar-satellite-reasoning
Branch 'init-project' set up to track remote branch 'init-project' from 'origin'.
Switched to a new branch 'init-project'
Already up to date.


<hr style="border:none;border-top:3px solid #d95f0e">
## 3 · Dependencies (statsmodels cho ARIMA)

In [3]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # set TRƯỚC khi torch chạm CUDA
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
!pip install -q timm torch_geometric h5py scikit-learn pyyaml tqdm statsmodels

torch 2.11.0+cu128 | CUDA: True
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 73.2 MB/s eta 0:00:00


<hr style="border:none;border-top:3px solid #d95f0e">
## 4 · Dữ liệu (mount Drive)
Phase A chỉ cần DKASC CSV; Himawari chỉ cần cho Phase B.

In [4]:
import pathlib
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = '/content/drive/MyDrive/tkg-solar-data'   # <-- ĐIỀN đúng thư mục Drive của bạn
DKASC_CSV    = f'{DATA_ROOT}/dkasc/alice_2020_2022_clean.csv'
HIMAWARI_DIR = f'{DATA_ROOT}/himawari_alice'
print(('OK  ' if pathlib.Path(DKASC_CSV).exists() else 'MISSING '), DKASC_CSV)
print(('OK  ' if pathlib.Path(f'{HIMAWARI_DIR}/frames.h5').exists() else '(Phase B mới cần) '), HIMAWARI_DIR)

Mounted at /content/drive
OK   /content/drive/MyDrive/tkg-solar-data/dkasc/alice_2020_2022_clean.csv
OK   /content/drive/MyDrive/tkg-solar-data/himawari_alice


<hr style="border:none;border-top:3px solid #d95f0e">
## 5 · Config + thiết bị
`resolve_runtime` tự chọn precision: **A100→bf16**, T4→fp16; auto-batch theo VRAM.

In [5]:
from src.training.config import Config
from src.training.gpu_autoscale import resolve_runtime
from src.common.seeding import seed_everything
from src.common.shapes import EMBED_DIM, FUSION_DIM, HORIZON_MINUTES, N_HORIZONS, N_METEO_FEATURES

cfg = Config.from_yaml('configs/paper_config.yaml')
cfg.device = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg.dkasc_csv = DKASC_CSV
cfg.himawari_dir = HIMAWARI_DIR
cfg.checkpoint_dir = f'{DATA_ROOT}/checkpoints'
cfg.cache_dir = 'data/cache'
resolve_runtime(cfg)
seed_everything(cfg.seed)
print('device', cfg.device, '| precision', cfg.precision, '| batch', cfg.batch_size)
print('contract', N_METEO_FEATURES, 'feats | split', cfg.train_end, '/', cfg.val_end, '| min_steps', cfg.min_steps)

[gpu_autoscale] VRAM~39GB -> batch=16 grad_accum=4 precision=bf16
device cuda | precision bf16 | batch 16
contract 4 feats | split 2021-10-01 / 2022-01-01 | min_steps 2000


<hr style="border:none;border-top:3px solid #d95f0e">
## 6 · Helper resume/skip checkpoint

In [ ]:
from pathlib import Path

AUTO_RESUME = True   # True: train xuyên đêm không hỏi input (done->skip, partial->resume)

def ckpt_status(label):
    p = Path(cfg.checkpoint_dir) / f'last_{label}.pt'
    if not p.exists():
        return 'none'
    try:
        ck = torch.load(p, map_location='cpu', weights_only=False)
    except Exception:
        return 'none'
    return 'done' if ck.get('done') else 'partial'

def decide(label):
    st = ckpt_status(label)
    if st == 'none':
        return 'fresh'
    if st == 'partial':
        print(f'[{label}] checkpoint dang dở -> resume'); return 'resume'
    if AUTO_RESUME:
        print(f'[{label}] đã train xong -> skip (dùng checkpoint)'); return 'skip'
    return 'fresh' if input(f"[{label}] xong. 'r'=train lại, Enter=bỏ qua: ").strip().lower() == 'r' else 'skip'

print('decide -> fresh|resume|skip | AUTO_RESUME =', AUTO_RESUME)

decide -> fresh|resume|skip | AUTO_RESUME = True


<hr style="border:none;border-top:4px solid #2c7fb8">

# 🟦 PHASE A — Baselines (KHÔNG cần Himawari)

`use_satellite=False` → PV+meteo full DKASC 2020–2022 (sat=0, `img_size=8`). Chạy ngon trên T4 free.

## A1 · Pipeline (PV + meteo)

In [7]:
from src.data_pipeline import DataPipeline

splits_base = DataPipeline.load(
    cfg.dkasc_csv, cfg.himawari_dir,
    k=cfg.k, batch_size=256, img_size=8,
    min_steps=cfg.min_steps, train_end=cfg.train_end, val_end=cfg.val_end,
    cadence_min=cfg.cadence_min, night_ghi_thresh=cfg.night_ghi_thresh,
    cache_dir=cfg.cache_dir, use_satellite=False, scaler_out=cfg.scaler_out,
)
for k_, v_ in splits_base.meta.items():
    print(f'  {k_:18s}: {v_}')

  source            : dkasc_alice
  pv_units          : kW
  use_satellite     : False
  n_meteo_features  : 4
  sat_channels      : 1
  img_size          : 8
  cadence_min       : 5
  k                 : 12
  n_steps           : 156801
  split_mode        : dates
  train_end         : 2021-10-01
  val_end           : 2022-01-01
  n_train_windows   : 88882
  n_val_windows     : 14509
  n_test_windows    : 53341


## A2 · Train 6 baselines
Persistence/ARIMA = eval-only. ARIMA tự subsample 2000 cửa sổ + thanh %. Checkpoint lệch (data cũ) tự bị xóa → train fresh.

In [8]:
import copy
from src.evaluation.baseline_models import build_baseline, is_eval_only
from src.lstm_baseline.lstm_forecaster import LSTMForecaster
from src.training.train_loop import fit
from src.training.evaluate import evaluate_model

bcfg = copy.copy(cfg); bcfg.batch_size = 256
# bcfg.epochs = 50
BASELINES = ['Persistence', 'ARIMA', 'LSTM', 'GRU', 'Transformer', 'Temporal-GNN']
# BASELINES = [m for m in BASELINES if m != 'ARIMA']
ARIMA_MAX_WINDOWS = 2000

# Nạp kết quả đã có trên Drive -> chạy lại cell KHÔNG tính lại model đã xong (kể cả ARIMA/Persistence).
import json
_BPATH = f'{DATA_ROOT}/outputs/base_results.json'
pathlib.Path(f'{DATA_ROOT}/outputs').mkdir(parents=True, exist_ok=True)
base_results = json.load(open(_BPATH)) if pathlib.Path(_BPATH).exists() else {}
FORCE = []   # tên model muốn tính LẠI (vd ['ARIMA']); để rỗng = dùng lại kết quả cũ
for _f in FORCE: base_results.pop(_f, None)
print('Đã có sẵn (sẽ bỏ qua):', list(base_results) or 'chưa có')

def build_base(name):
    return LSTMForecaster(hidden_dim=bcfg.embed_dim, dropout=bcfg.dropout) if name == 'LSTM' else build_baseline(name, bcfg)

def run_base(name):
    if name in base_results:
        print(f'== {name}: đã có kết quả -> bỏ qua =='); return
    import gc; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    seed_everything(bcfg.seed)
    model_b = build_base(name)
    if is_eval_only(model_b):
        print(f'== {name} (eval-only) =='); model_b.to(bcfg.device)
    else:
        action = decide(name)
        loaded = False
        if action == 'skip':
            try:  # checkpoint cũ (data 7-feature) lệch shape -> xóa + train fresh
                best = Path(bcfg.checkpoint_dir) / f'best_{name}.pt'
                model_b.load_state_dict(torch.load(best, map_location=bcfg.device, weights_only=True)['model_state'])
                model_b.to(bcfg.device); loaded = True
            except Exception as e:
                print(f'[{name}] checkpoint cũ không khớp ({type(e).__name__}) -> xóa, train fresh')
                for kind in ('best', 'last'):
                    (Path(bcfg.checkpoint_dir) / f'{kind}_{name}.pt').unlink(missing_ok=True)
        if not loaded:
            if action == 'fresh':
                (Path(bcfg.checkpoint_dir) / f'last_{name}.pt').unlink(missing_ok=True)
            try:
                fit(model_b, splits_base, bcfg, verbose=True, desc=name, resume=(action != 'fresh'))
            except RuntimeError as e:  # resume vào checkpoint lệch (data cũ) -> xóa, train fresh
                if 'state_dict' in str(e) or 'size mismatch' in str(e):
                    print(f'[{name}] resume vào checkpoint lệch -> xóa, train fresh')
                    for kind in ('best', 'last'):
                        (Path(bcfg.checkpoint_dir) / f'{kind}_{name}.pt').unlink(missing_ok=True)
                    model_b = build_base(name)
                    fit(model_b, splits_base, bcfg, verbose=True, desc=name, resume=False)
                else:
                    raise
    mw = ARIMA_MAX_WINDOWS if name == 'ARIMA' else None
    m = evaluate_model(model_b, splits_base.test_loader, splits_base.scalers, bcfg.device,
                       bcfg.mape_min_value, max_windows=mw, progress=(name == 'ARIMA'))
    base_results[name] = m['overall']
    json.dump(base_results, open(_BPATH, 'w'), indent=2)   # lưu NGAY sau mỗi model -> chạy lại không mất
    print(f"{name}: MAE={m['overall']['mae']:.4f} RMSE={m['overall']['rmse']:.4f} MAPE={m['overall']['mape']:.2f}%")
    model_b.to('cpu'); del model_b

for _n in BASELINES:
    run_base(_n)

Đã có sẵn (sẽ bỏ qua): ['Persistence', 'ARIMA', 'LSTM', 'GRU', 'Transformer', 'Temporal-GNN']
== Persistence: đã có kết quả -> bỏ qua ==
== ARIMA: đã có kết quả -> bỏ qua ==
== LSTM: đã có kết quả -> bỏ qua ==
== GRU: đã có kết quả -> bỏ qua ==
== Transformer: đã có kết quả -> bỏ qua ==
== Temporal-GNN: đã có kết quả -> bỏ qua ==


In [ ]:
import json
pathlib.Path(f'{DATA_ROOT}/outputs').mkdir(parents=True, exist_ok=True)
json.dump(base_results, open(f'{DATA_ROOT}/outputs/base_results.json', 'w'), indent=2)
print('Đã lưu', len(base_results), 'baselines -> Drive/outputs/base_results.json')

Đã lưu 6 baselines -> Drive/outputs/base_results.json


## A3 · Bảng xếp hạng baselines (kW)

In [ ]:
from src.evaluation.benchmark_table import build_benchmark, relative_ordering
print(build_benchmark(base_results, scalers=splits_base.scalers))
print('\nThứ tự (tốt→kém MAE):', ' < '.join(relative_ordering(base_results)))

| Model | MAE [0,1] | MAE (paper) | RMSE [0,1] | RMSE (paper) | MAPE% | MAPE% (paper) | MAE (kW) | RMSE (kW) |
|-------|-----------|-------------|------------|--------------|-------|---------------|----------|-----------|
| Persistence | 0.088 | - | 0.132 | - | 41.19 | - | 19.17 | 28.60 |
| ARIMA | 0.076 | - | 0.136 | - | 34.53 | - | 16.41 | 29.42 |
| LSTM | 0.063 | 0.128 | 0.102 | 0.201 | 28.15 | 9.84 | 13.64 | 22.00 |
| GRU | 0.055 | 0.121 | 0.097 | 0.192 | 25.92 | 9.17 | 11.86 | 21.00 |
| Transformer | 0.065 | 0.109 | 0.106 | 0.174 | 29.07 | 7.96 | 13.99 | 23.02 |
| Temporal-GNN | 0.051 | 0.097 | 0.093 | 0.161 | 23.57 | 7.11 | 11.11 | 20.25 |

Thứ tự (tốt→kém MAE): Temporal-GNN < GRU < LSTM < Transformer < ARIMA < Persistence


<hr style="border:none;border-top:4px solid #d95f0e">

# 🟧 PHASE B — Proposed (TKG) — cần Himawari

Chỉ chạy khi có `himawari_alice/frames.h5`. ViT-224 → nên A100. T4 thì bật `cfg.freeze_backbone=True`.

> Himawari một phần → split cố định rỗng: dùng `train_frac=.7, val_frac=.15` (bỏ train_end/val_end) + hạ min_steps.

## B1 · Pipeline (PV + meteo + vệ tinh)

In [11]:
import time
assert pathlib.Path(f'{HIMAWARI_DIR}/frames.h5').exists(), 'Thiếu frames.h5 -> upload Himawari trước'
# freeze_backbone đã bật sẵn trong paper_config.yaml (ViT đông cứng -> chống overfit).
# img_size=64: GIỮ frame gốc (nhẹ ~0.7GB), encoder tự upsample 64->224 cho ViT mỗi batch.
# (Đừng dùng cfg.img_size=224 ở đây -> sẽ resize cả mảng ~10GB, load 30+ phút.)
print('Đang load + align DKASC + Himawari (lần đầu ~2-4 phút)...', flush=True)
_t = time.time()
splits_full = DataPipeline.load(
    cfg.dkasc_csv, cfg.himawari_dir,
    k=cfg.k, batch_size=cfg.batch_size, img_size=64,
    min_steps=cfg.min_steps, train_end=cfg.train_end, val_end=cfg.val_end,
    cadence_min=cfg.cadence_min, night_ghi_thresh=cfg.night_ghi_thresh,
    cache_dir=cfg.cache_dir, use_satellite=True, scaler_out=cfg.scaler_out,
)
print(f'Xong sau {time.time()-_t:.0f}s |', splits_full.meta)

Đang load + align DKASC + Himawari (lần đầu ~2-4 phút)...
Xong sau 35s | {'source': 'dkasc_alice', 'pv_units': 'kW', 'use_satellite': True, 'n_meteo_features': 4, 'sat_channels': 1, 'img_size': 64, 'cadence_min': 5, 'k': 12, 'n_steps': 136399, 'split_mode': 'dates', 'train_end': '2021-10-01', 'val_end': '2022-01-01', 'n_train_windows': 80291, 'n_val_windows': 12044, 'n_test_windows': 43995}


## B2 · Train Proposed (TKGSolarModel)

In [ ]:
from src.fusion_predictor.tkg_solar_model import TKGSolarModel

model = TKGSolarModel.from_config(cfg)
_tot = sum(p.numel() for p in model.parameters())
_tr  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('params', f'{_tot:,}', '| trainable', f'{_tr:,}', '| FUSION_DIM', FUSION_DIM)
print('freeze_backbone =', cfg.freeze_backbone, '| weight_decay', cfg.weight_decay, '| lr_scheduler', cfg.lr_scheduler)

action = decide('tkg')
history = None
if action == 'skip':
    try:  # checkpoint cũ lệch -> xóa, train fresh
        ck = torch.load(Path(cfg.checkpoint_dir) / 'last_tkg.pt', map_location=cfg.device, weights_only=False)
        model.load_state_dict(ck['model_state']); model.to(cfg.device)
        history = ck['history']; history['best_val_mae'] = ck['best_val']
        history['best_checkpoint'] = str(Path(cfg.checkpoint_dir) / 'best_tkg.pt')
    except Exception as e:
        print(f'[tkg] checkpoint cũ không khớp ({type(e).__name__}) -> xóa, train fresh')
        for kind in ('best', 'last'):
            (Path(cfg.checkpoint_dir) / f'{kind}_tkg.pt').unlink(missing_ok=True)
if history is None:
    if action == 'fresh':
        (Path(cfg.checkpoint_dir) / 'last_tkg.pt').unlink(missing_ok=True)
    try:
        history = fit(model, splits_full, cfg, verbose=True, desc='tkg', resume=(action != 'fresh'))
    except RuntimeError as e:  # resume vào checkpoint lệch (data cũ) -> xóa, train fresh
        if 'state_dict' in str(e) or 'size mismatch' in str(e):
            print(f'[tkg] resume vào checkpoint lệch -> xóa, train fresh')
            for kind in ('best', 'last'):
                (Path(cfg.checkpoint_dir) / f'{kind}_tkg.pt').unlink(missing_ok=True)
            model = TKGSolarModel.from_config(cfg)
            history = fit(model, splits_full, cfg, verbose=True, desc='tkg', resume=False)
        else:
            raise
print('\nbest val-MAE:', round(history['best_val_mae'], 5))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

params 86,917,769 | FUSION_DIM 384
[full] checkpoint dang dở -> resume
full: resume from epoch 3 (best val_mae=13.52477)


full ep 4/200:   0%|          | 0/5019 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: Flash Attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:124.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


## B3 · Đánh giá Proposed (theo horizon, kW)

In [ ]:
from src.training.train_loop import predict_loader
from src.metrics.regression_metrics import compute_per_horizon

yt, yp = predict_loader(model, splits_full.test_loader, cfg.device)
yt = splits_full.scalers.inverse_pv(yt.numpy()); yp = splits_full.scalers.inverse_pv(yp.numpy())
per = compute_per_horizon(yt, yp, HORIZON_MINUTES, cfg.mape_min_value)
print(f"{'horizon':>10} | {'MAE':>9} | {'RMSE':>9} | {'MAPE %':>9}")
print('-' * 46)
for lab in ['overall', *[f'{m}min' for m in HORIZON_MINUTES]]:
    mm = per[lab]
    print(f"{lab:>10} | {mm['mae']:9.4f} | {mm['rmse']:9.4f} | {mm['mape']:9.2f}")

In [ ]:
import json
pathlib.Path(f'{DATA_ROOT}/outputs').mkdir(parents=True, exist_ok=True)
json.dump(per['overall'], open(f'{DATA_ROOT}/outputs/proposed_result.json', 'w'), indent=2)
print('Đã lưu Proposed -> Drive/outputs/proposed_result.json')

## B4 · Bảng tổng (ghép baselines + Proposed từ Drive) + ablation

In [ ]:
import json
from src.evaluation.benchmark_table import build_benchmark, relative_ordering, build_ablation_table

out = f'{DATA_ROOT}/outputs'
all_results = {}
if pathlib.Path(f'{out}/base_results.json').exists():
    all_results.update(json.load(open(f'{out}/base_results.json')))
if pathlib.Path(f'{out}/proposed_result.json').exists():
    all_results['Proposed'] = json.load(open(f'{out}/proposed_result.json'))
assert all_results, 'Chưa có kết quả trên Drive -> chạy Phase A và/hoặc Phase B trước'

_sc = (splits_full if 'splits_full' in globals() else splits_base).scalers
print(build_benchmark(all_results, scalers=_sc))
print('\nThứ tự (tốt→kém MAE):', ' < '.join(relative_ordering(all_results)))

RUN_ABLATION = False   # True -> ±sat/±graph/±meteo (cần splits_full + ~3-4 lần train Proposed)
if RUN_ABLATION and 'splits_full' in globals():
    abl = {}
    for label, ov in [('Full', {}), ('-sat', {'use_sat': False}),
                      ('-graph', {'use_graph': False}), ('-meteo', {'use_meteo': False})]:
        acfg = copy.copy(cfg); acfg.use_sat = acfg.use_meteo = acfg.use_graph = True
        for kk, vv in ov.items(): setattr(acfg, kk, vv)
        seed_everything(acfg.seed)
        am = TKGSolarModel.from_config(acfg)
        fit(am, splits_full, acfg, verbose=True, desc=f'abl-{label}', resume=False)
        abl[label] = evaluate_model(am, splits_full.test_loader, splits_full.scalers,
                                    acfg.device, acfg.mape_min_value)['overall']
        am.to('cpu'); del am
    print(build_ablation_table(abl))

<hr style="border:none;border-top:3px solid #d95f0e">
## 7 · Lưu checkpoint Proposed về Drive/outputs

In [ ]:
import shutil
SAVE_DIR = f'{DATA_ROOT}/outputs'
pathlib.Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
if 'history' in globals() and history and pathlib.Path(history['best_checkpoint']).exists():
    shutil.copy(history['best_checkpoint'], SAVE_DIR)
if pathlib.Path(cfg.scaler_out).exists():
    shutil.copy(cfg.scaler_out, SAVE_DIR)
cfg.save(f'{SAVE_DIR}/resolved_config.yaml')
print('Đã lưu ->', SAVE_DIR)